In [3]:
!pip install tavily-python
from tavily import TavilyClient
tavily_client = TavilyClient(api_key="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")

In [4]:
from langchain_tavily import TavilySearch
import pandas as pd


search_tool = TavilySearch(
    max_results = 5,
    topic = "general",
    tavily_api_key = "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",
)

In [3]:
search_tool.invoke("Quand les urgences du centre hospitalier de Fougères (France) ont-elles partiellement fermées en 2025 ? Ne répond que par une liste [date, raison de la fermeture]. Par exemple : 2025-05-18: Fermeture partielle nocturne, 2025-09-01: Régulation temporaire")

{'query': 'Quand les urgences du centre hospitalier de Fougères (France) ont-elles partiellement fermées en 2025 ? Ne répond que par une liste [date, raison de la fermeture]. Par exemple : 2025-05-18: Fermeture partielle nocturne, 2025-09-01: Régulation temporaire',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'title': 'A. RESULTS OF 91, 182 & 364 DAYS TREASURY BILLS ISSUES 2659/ ...',
   'url': 'https://www.centralbank.go.ke/uploads/182_day_historical_treasury_bill_results/904254459_RESULTS+2659-091+2633-182+2588-364+DATED+08-12-2025.pdf',
   'content': 'A. RESULTS OF 91, 182 & 364 DAYS TREASURY BILLS ISSUES 2659/091, 2633/182 & 2588/364 DATED 08-12-2025',
   'score': 0.10946776,
   'raw_content': None},
  {'title': 'ANDRÉ MARSIGLIA | PÂNICO - 08/12/2025 - YouTube A. RESULTS OF 91, 182 & 364 DAYS TREASURY BILLS ISSUES 2659/ ... Fixtures & results | UEFA Champions League 2025/26',
   'url': 'https://www.youtube.com/watch?v=jlcYTW-2hgU',
   'content': '20 

In [4]:
def search_er_closures_for_hospital(
    hospital_name: str,
    city: str | None = None,
    year_from: int = 2023,
    year_to: int = 2025,
):
    """
    Usa Tavily para buscar artículos sobre cierres / regulación de urgencias
    para un hospital concreto.
    Devuelve una lista de dicts con título, url, snippet, etc.
    """
    years = f"{year_from}-{year_to}" if year_from != year_to else str(year_from)

    # Query pensada para noticias francesas
    query_parts = [
        f'fermeture urgences "{hospital_name}"',
        f"{years}",
        "site:fr",
    ]
    if city:
        query_parts.insert(1, city)

    query = " ".join(query_parts)
    print("Query enviada a Tavily:\n", query)

    results = search_tool.invoke(query)  # Tavily devuelve lista de dicts
    return results

In [6]:
results = search_er_closures_for_hospital(
    hospital_name="Centre hospitalier de Fougères",
    city="Fougères",
    year_from=2023,
    year_to=2025,
)


Query enviada a Tavily:
 fermeture urgences "Centre hospitalier de Fougères" Fougères 2023-2025 site:fr


In [14]:
results.keys()
import json
print(json.dumps(results, indent=2, ensure_ascii=False))

{
  "query": "fermeture urgences \"Centre hospitalier de Fougères\" Fougères 2023-2025",
  "follow_up_questions": null,
  "answer": null,
  "images": [],
  "results": [
    {
      "url": "https://fougeres.fr/wp-content/uploads/2024/10/PV-du-11-mai-2023.pdf",
      "title": "PROCES-VERBAL Conseil Municipal du jeudi 11 mai ...",
      "content": "La Ville de Fougères, Fougères Habitat et le Centre Hospitalier de Fougères s'engagent : - à fournir tous les documents/données nécessaires",
      "score": 0.14160188,
      "raw_content": null
    },
    {
      "url": "https://sante.gouv.fr/fichiers/bo/2023/2023.23.sante.pdf",
      "title": "Protection sociale - Solidarité n° 2023/23 du 15 décembre 2023",
      "content": "En matière d'opérations immobilières tertiaires relevant du budget de gestion, en cas d'absence ou d'empêchement du/de la directeur(rice) délégué(e) aux opérations, délégation de signature est accordée à Mme Carole BLANC pour signer : • la notification aux organismes des 

In [2]:
df_results = pd.DataFrame(results)
df_results[["title", "url", "content"]].head()

NameError: name 'pd' is not defined

Links por región:

	1.	Auvergne-Rhône-Alpes: 
        https://www.sfmu.org/fr/actualites/actualites-de-l-urgences/auvergne-rhone-alpes-14-arretes-pris-sur-la-regulation-nocturne-de-structures-des-urgences/new_id/69979
        
	2.	Bourgogne-Franche-Comté  ￼
	3.	Bretagne (Bretaña):  https://www.bretagne.ars.sante.fr/arretes-portant-regulation-temporaire-de-lacces-aux-urgences
	4.	Centre-Val de Loire  ￼
	5.	Corse (Córcega)  ￼
	6.	Grand Est  ￼
	7.	Hauts-de-France  ￼
	8.	Île-de-France  ￼
	9.	Normandie (Normandía)  ￼
	10.	Nouvelle-Aquitaine (Nueva Aquitania)  ￼
	11.	Occitanie  ￼
	12.	Pays de la Loire  ￼
	13.	Provence-Alpes-Côte d’Azur  ￼


### Broad Search

In [5]:

tavily_client = TavilyClient(api_key="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")


queries = [
  '("régulation temporaire" OR "accès régulé") urgences arrêté',
  '("fermeture temporaire" OR "accès aux urgences") ARS arrêté',
  '"arrêté" "service des urgences" "accès régulé"',
]

all_results = []
for q in queries:
    resp = tavily_client.search(
        query=q,
        search_depth="advanced",
        max_results=10,
        include_answer=False,
        include_raw_content=False,
        include_domains=["ars.sante.fr"]  # portal + often links out
    )
    all_results.append(resp)


print(all_results)

[{'query': '(régulation temporaire OR accès régulé) urgences arrêté', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.occitanie.ars.sante.fr/media/135518/download?inline', 'title': 'Arrêté ARS Occitanie n°2025-1088 fixant la régulation ...', 'content': 'relatif à la régulation temporaire de l’accès aux urgences ; Vu le courrier du directeur du CH de MONTAUBAN en date du 10 janvier 2025 demandant l’autorisation de réguler de façon temporaire l’accès aux urgences de son établissement de santé ; Considérant que malgré les efforts de recrutements et de mobilisation de l’intérim mis en œuvre par le Centre hospitalier, l’établissement ne parvient pas à réunir les effectifs nécessaires à une couverture totale des plannings ; Considérant les [...] Arrêté ARS Occitanie n°2025-1088 fixant la régulation temporaire de l’accès aux urgences du Centre Hospitalier de MONTAUBAN Le directeur général de l’agence régionale de santé Occitanie, Vu le code de la sa

In [6]:
for i, resp in enumerate(all_results):
    print("=" * 80)
    print(f"QUERY {i+1}: {resp['query']}")
    print("-" * 80)

    for j, r in enumerate(resp["results"]):
        print(f"[{j+1}] {r['title']}")
        print(f"URL: {r['url']}")
        print(f"Score: {r['score']}")
        print(f"Snippet: {r['content'][:300]}")  # truncate
        print()

QUERY 1: (régulation temporaire OR accès régulé) urgences arrêté
--------------------------------------------------------------------------------
[1] Arrêté ARS Occitanie n°2025-1088 fixant la régulation ...
URL: https://www.occitanie.ars.sante.fr/media/135518/download?inline
Score: 0.8335554
Snippet: relatif à la régulation temporaire de l’accès aux urgences ; Vu le courrier du directeur du CH de MONTAUBAN en date du 10 janvier 2025 demandant l’autorisation de réguler de façon temporaire l’accès aux urgences de son établissement de santé ; Considérant que malgré les efforts de recrutements et de

[2] Arrêtés portant régulation temporaire de l'accès aux urgences
URL: https://www.bretagne.ars.sante.fr/arretes-portant-regulation-temporaire-de-lacces-aux-urgences
Score: 0.8215175
Snippet: Arrêté n°2025/224 -  modifiant l'arrêté n°2025/209 - régulation temporaire de l’accès aux urgences du Centre Hospitalier de Guingamp- à compter du 12 juillet jusqu'au 25 août - 18h30 à 8h30
 Arrêté n°202

Trying some things out

1) Use Map to discover “where the content lives” per region


In [ ]:


tavily_client = TavilyClient(api_key="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")
response = tavily_client.map("https://www.ars.sante.fr/")

print(response)

{'base_url': 'https://www.ars.sante.fr/', 'results': ['https://www.ars.sante.fr/', 'https://www.ars.sante.fr/plandusite', 'https://www.ars.sante.fr/contactez-votre-ars', 'https://www.ars.sante.fr/quest-ce-que-la-democratie-en-sante-3', 'https://www.ars.sante.fr/donnees-personnelles', 'https://www.ars.sante.fr/application-recosante-agir-pour-proteger-votre-sante-0', 'https://www.ars.sante.fr/la-mission-inspection-controle', 'https://www.ars.sante.fr/les-contrats-locaux-de-sante', 'https://www.ars.sante.fr/la-gouvernance-des-agences-regionales-de-sante', 'https://www.ars.sante.fr/le-service-sanitaire-des-etudiants-en-sante', 'https://www.ars.sante.fr/liste-appels-projet-candidature-nationale', 'https://www.ars.sante.fr/les-ars-recrutent', 'https://www.ars.sante.fr/les-actus-des-ars-0', 'https://www.ars.sante.fr/les-projets-regionaux-de-sante-2018-2028-revises-en-2023', 'https://www.ars.sante.fr/les-principaux-partenaires-des-agences-regionales-de-sante', 'https://www.ars.sante.fr/les-con

Use the Crawl to find the links with relevant information for the closures

In [16]:


queries = [
  '("régulation temporaire" OR "accès régulé") urgences arrêté',
  '("fermeture temporaire" OR "accès aux urgences") ARS arrêté',
  '"arrêté" "service des urgences" "accès régulé"', 
]

tavily_client = TavilyClient(api_key="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")
response = tavily_client.crawl("https://www.auvergne-rhone-alpes.ars.sante.fr/", instructions="Find all pages with information on régulation temporaire OR accès régulé OR fermeture temporaire OR arrêté OR accès régulé of urgences in 2023")

print(response)

{'base_url': 'https://www.auvergne-rhone-alpes.ars.sante.fr/', 'results': [{'url': 'https://www.auvergne-rhone-alpes.ars.sante.fr/gestion-de-crise-et-situations-exceptionnelles-0', 'raw_content': '* [Aller au menu principal,](#main-menu)\n* [Aller au contenu](#main-content)\n\n* [Twitter](https://twitter.com/ARS_ARA_SANTE "Twitter  (nouvelle fenêtre)")\n* [LinkedIn](https://www.linkedin.com/company/agence-r%C3%A9gionale-de-sant%C3%A9-auvergne-rh%C3%B4ne-alpes "LinkedIn (nouvelle fenêtre)")\n* [Facebook](https://www.facebook.com/profile.php?id=100089357238189 "Facebook (nouvelle fenêtre)")\n* [YouTube](https://www.youtube.com/@arsauvergne-rhone-alpes "YouTube (nouvelle fenêtre)")\n\nL\'ARS Auvergne-Rhône-Alpes est là\n\n\n\n# Gestion de crise et situations exceptionnelles\n\n* Autoriser\n* Autoriser\n* Autoriser\n* Autoriser\n\n* [Infections respiratoires aigües (IRA) : mesures à appliquer](/infections-respiratoires-aigues-ira-procedure-de-gestion-au-sein-des-etablissements-medico-socia

### Auvergne-Rhône-Alpes

In [ ]:

response = tavily_client.search("Who is Leo Messi?")

print(response)

In [19]:

tavily_client = TavilyClient(api_key="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")


queries = [
  '("régulation temporaire" OR "accès régulé") urgences arrêté CH BOURG EN BRESSE',
  '("fermeture temporaire" OR "accès aux urgences") ARS arrêté CH BOURG EN BRESSE',
  '"arrêté" "service des urgences" "accès régulé" CH BOURG EN BRESSE',
]

all_results = []
for q in queries:
    resp = tavily_client.search(
        query=q,
        search_depth="advanced",
        max_results=20,
        include_answer=False,
        include_raw_content=False,
        include_domains=["https://ars.sante.fr/"]  # portal + often links out
    )
    all_results.append(resp)


print(all_results)

[{'query': '(régulation temporaire OR accès régulé) urgences arrêté CH BOURG EN BRESSE', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.auvergne-rhone-alpes.ars.sante.fr/index.php/media/145570/download?inline', 'title': "Comité Régional d'Allocation des Ressources", 'content': "Avis n°2025-04 du 26 septembre 2025 sur l’arrêté de régulation temporaire d’accès aux Services des Urgences Le décret n°2023-1374 du 29 décembre 2023, relatif aux conditions d'implantation de l'activité de médecine d'urgence, fixe, à travers l’article R.6123-18-2 du Code de la Santé Publique (CSP) 2023, les modalités de régulation temporaire et/ou pérenne de l’accès aux urgences. Ainsi, à titre temporaire et lorsque les circonstances locales le justifient, les établissements disposant d'une structure d’urgences ou d'une antenne de médecine d'urgence peuvent, par arrêté du 2 juillet 2024 du directeur général de l'Agence Régionale de Santé (ARS), organiser un accès régu

In [20]:
for i, resp in enumerate(all_results):
    print("=" * 80)
    print(f"QUERY {i+1}: {resp['query']}")
    print("-" * 80)

    for j, r in enumerate(resp["results"]):
        print(f"[{j+1}] {r['title']}")
        print(f"URL: {r['url']}")
        print(f"Score: {r['score']}")
        print(f"Snippet: {r['content'][:300]}")  # truncate
        print()

QUERY 1: (régulation temporaire OR accès régulé) urgences arrêté CH BOURG EN BRESSE
--------------------------------------------------------------------------------
[1] Comité Régional d'Allocation des Ressources
URL: https://www.auvergne-rhone-alpes.ars.sante.fr/index.php/media/145570/download?inline
Score: 0.7251239
Snippet: Avis n°2025-04 du 26 septembre 2025 sur l’arrêté de régulation temporaire d’accès aux Services des Urgences Le décret n°2023-1374 du 29 décembre 2023, relatif aux conditions d'implantation de l'activité de médecine d'urgence, fixe, à travers l’article R.6123-18-2 du Code de la Santé Publique (CSP) 2

[2] Groupement Hospitalier de Territoire Bresse Haut Bugey ...
URL: https://www.auvergne-rhone-alpes.ars.sante.fr/media/2060/download?inline
Score: 0.542161
Snippet: Les urgences (nuits, WE et jours fériés) sont transférées au CH de Bourg en Bresse sauf demande explicite du patient.Les patients ayant bénéficié d'une

[3] Arrêté n°2021-17-0110
URL: https://www.auvergn

### Crawling

In [ ]:
# Step 2. Defining the starting URL
start_url = "https://www.ars.sante.fr/"

# Step 3. Executing the crawl request with instructions to surface only pages about citrus fruits
response = tavily_client.crawl(
    url=start_url,
    max_depth=5,
    limit=50,
    instructions="Find all official administrative orders (arrêtés) regarding 'CH BOURG EN BRESSE'. Specifically look for pages about 'régulation temporaire', 'accès régulé', 'fermeture temporaire', or 'accès aux urgences' for the emergency department (urgences)."
)

# Step 4. Printing pages matching the query
for result in response["results"]:
    print(f"URL: {result['url']}")
    print(f"Snippet: {result['raw_content'][:200]}...\n")
